# Week 3 - Option 2: Implement Steering & Intervention

In this notebook, we move from observing bias to actively mitigating it. We will use the Jacobian Lens to find the direction in the latent space that corresponds to a stereotyped token, and then use a PyTorch hook to **steer** the model away from that direction during generation.

**Note for Google Colab Users:** Run the first cell to clone the necessary repositories and install dependencies.

In [1]:
# Colab Setup: Clone repositories and install dependencies
import os

# Install Jacobian Lens
if not os.path.exists('jacobian-lens'):
    !git clone https://github.com/anthropics/jacobian-lens.git
    %cd jacobian-lens
    !pip install -e .
else:
    %cd jacobian-lens

Cloning into 'jacobian-lens'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 67 (delta 0), reused 0 (delta 0), pack-reused 65 (from 2)
Receiving objects: 100% (67/67), 1.90 MiB | 4.49 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/jacobian-lens
Obtaining file:///content/jacobian-lens
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jlens (pyproject.toml) ... done
  Created wheel for jlens: filename=jlens-0.1.0-0.editable-py3-none-any.whl size=8901 sha256=1cfc936e37753f3dcbd8bcf5b49039126bf3f22c075cc571476bd8d205df0002
  Stored in directory: /tmp/pip-ephem-wheel-cache-1bwo7bjq/wheels/9b/16/f6/ff5117e12d375117559a1ded186e4d458b172d145efd7f032b
Successfully built jlens


In [2]:
import jlens
import torch
import transformers
import matplotlib.pyplot as plt
import seaborn as sns
from jlens.vis import compute_slice

jlens.configure_logging()

## 1. Load the Model and Pre-fitted Lens

In [3]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

print(f"Loading Model: {MODEL_NAME}...")
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)

print("Loading Lens...")
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
print("Done!")

Loading Model: Qwen/Qwen3.5-4B...


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

Loading Lens...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Done!


## 2. Baseline Generation (Biased)
Let's test the model on a prompt known to elicit gender bias in language models due to occupational stereotypes.

In [4]:
#prompt = "The nurse took a break from the busy shift, and then"
prompt = "The nurse was exhausted from the busy shift, so"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating Baseline (No Steering)...")
# We generate 10 tokens to see how the model continues the sentence
outputs = hf_model.generate(**inputs, max_new_tokens=10, do_sample=False)
baseline_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\nBaseline Output:\n{baseline_text}")

# We expect the model to likely use 'she' here.

Generating Baseline (No Steering)...

Baseline Output:
The nurse was exhausted from the busy shift, so she decided to take a nap. She was in


## 3. Intervention: Steering the Latent Space
We want to suppress the token `she` and optionally boost `he` or `they`. We can do this by using a PyTorch Forward Hook to literally subtract a steering vector from the model's residual stream at a specific layer (e.g., in the middle of the 'Workspace Band' we discovered in Week 2).

In [9]:
# 1. Identify the token IDs we want to suppress
target_token = " she"
target_token_id = tokenizer.encode(target_token)[0]

print(f"Target Token: '{target_token}' (ID: {target_token_id})")

# 2. Define the Steering Hook
STEERING_LAYER = 18 # Target the end of the Workspace Band
STEERING_ALPHA = 15 # The strength of our intervention

def steering_hook(module, input, output):
    # Check if the output is a tuple or a raw tensor
    if isinstance(output, tuple):
        hidden_states = output[0]
    else:
        hidden_states = output

    # Generate our steering vector from the embedding matrix
    steering_vector = hf_model.model.embed_tokens.weight[target_token_id]

    # Apply steering to the final sequence position (hidden_states is 3D)
    hidden_states[:, -1, :] -= STEERING_ALPHA * steering_vector

    # Return the modified states in the exact format they were received
    if isinstance(output, tuple):
        return (hidden_states,) + output[1:]
    else:
        return hidden_states


# 3. Register the hook
handle = hf_model.model.layers[STEERING_LAYER].register_forward_hook(steering_hook)
print(f"Steering hook registered at Layer {STEERING_LAYER} with Alpha {STEERING_ALPHA}")

Target Token: ' she' (ID: 1292)
Steering hook registered at Layer 18 with Alpha 15


In [11]:
# 1. Identify the contrastive tokens
token_she = tokenizer.encode(" she")[0]
token_he = tokenizer.encode(" he")[0]

print(f"' she' ID: {token_she}, ' he' ID: {token_he}")

# 2. Define the Contrastive Steering Vector using the Output Head (lm_head)
# We calculate the vector that points AWAY from "she" and TOWARDS "he"
steering_vector = hf_model.lm_head.weight[token_he] - hf_model.lm_head.weight[token_she]

# 3. Define the Steering Hook
STEERING_LAYER = 28 # Strike closer to the output so the model can't recover
STEERING_ALPHA = 2.0 # You can tweak this up or down (try 1.0 to 5.0)

def steering_hook(module, input, output):
    if isinstance(output, tuple):
        hidden_states = output[0]
    else:
        hidden_states = output

    # ADD the contrastive vector to push the hidden state towards "he"
    hidden_states[:, -1, :] += STEERING_ALPHA * steering_vector

    if isinstance(output, tuple):
        return (hidden_states,) + output[1:]
    else:
        return hidden_states

# 4. Register the hook
handle = hf_model.model.layers[STEERING_LAYER].register_forward_hook(steering_hook)
print(f"Contrastive hook registered at Layer {STEERING_LAYER} with Alpha {STEERING_ALPHA}")


' she' ID: 1292, ' he' ID: 551
Contrastive hook registered at Layer 28 with Alpha 2.0


In [13]:
# 1. Identify the contrastive tokens
token_she = tokenizer.encode(" she")[0]
token_he = tokenizer.encode(" he")[0]

print(f"' she' ID: {token_she}, ' he' ID: {token_he}")

# 2. Define the Contrastive Steering Vector using the Output Head
steering_vector = hf_model.lm_head.weight[token_he] - hf_model.lm_head.weight[token_she]

# 3. Define the Steering Hook on the FINAL Norm
STEERING_ALPHA = 15.0 # Let's hit it with a sledgehammer to prove it works

def steering_hook(module, input, output):
    # The output of the final norm is a single tensor: (batch, seq_len, hidden_size)
    hidden_states = output

    # Push the representation towards "he" exactly at the last token position
    hidden_states[:, -1, :] += STEERING_ALPHA * steering_vector

    return hidden_states

# 4. Register the hook on the absolute final normalization step
handle = hf_model.model.norm.register_forward_hook(steering_hook)
print(f"Contrastive hook registered on Final Norm with Alpha {STEERING_ALPHA}")


' she' ID: 1292, ' he' ID: 551
Contrastive hook registered on Final Norm with Alpha 15.0


## 4. Evaluate the Intervention
Now we run the exact same generation. The hook will intercept the forward pass, subtract our steering vector, and ideally force the model to generate a non-stereotypical continuation.

In [14]:
print("Generating with Intervention...")

steered_outputs = hf_model.generate(**inputs, max_new_tokens=10, do_sample=False)
steered_text = tokenizer.decode(steered_outputs[0], skip_special_tokens=True)

print(f"\nSteered Output:\n{steered_text}")

# Remember to remove the hook when done!
handle.remove()
print("\nHook removed.")

Generating with Intervention...

Steered Output:
The nurse was exhausted from the busy shift, so he decided to take a 15-minute nap

Hook removed.


### Experimentation
Try changing `STEERING_ALPHA`. If it is too low, the bias remains. If it is too high, the model's generation might degrade into gibberish (coherence loss). Finding the "Goldilocks zone" is the core challenge of steering!

### Experimentation & Analysis: The "Whac-A-Mole" of Steering

During this experiment, we discovered that simply identifying a biased representation isn't enough to easily fix it. Our intervention took three iterations to finally succeed, which taught us valuable lessons about how modern LLMs process information:

**Attempt 1: The Input Embedding Disconnect**
We initially tried to suppress the token by extracting its vector from the input embedding matrix (`embed_tokens.weight`) and subtracting it at Layer 18. This completely failed. We learned that by Layer 18, the model has transformed the representations so deeply that the original "input" vector is meaningless noise to the network.

**Attempt 2: The Self-Correction of Deep Layers**
We then switched to the correct output vector (`lm_head.weight`) and used a contrastive approach (subtracting "she", adding "he") at Layer 28. Surprisingly, the model still outputted "she"! We discovered that because the model still had layers 29, 30, and 31 left, the remaining attention heads and MLPs recognized our intervention as an anomaly and actively "projected it away" to restore the heavy bias established in the earlier layers.

**Attempt 3: The Un-fightable Final Norm**
To finally succeed, we had to intervene on the absolute last step of the model: the final `LayerNorm` (`model.norm`), effectively injecting our contrastive vector a split-second before the logits are calculated. By doing this, we stripped the model of any remaining layers it could use to fight back. With an Alpha of `15.0`, we successfully overwhelmed the model's internal probability distribution, forcing the biased prompt (*"The nurse... so"*) to confidently generate the counter-stereotypical pronoun `"he"`.

**Conclusion:**
Bias runs deep within the network's layers. While the "Global Workspace" (middle layers) is where the bias is most visible, the final layers actively enforce it. Steering requires striking as close to the output as possible with contrastive vectors to prevent the model from self-correcting.
